##1. Importing the required libraries

In [10]:
pip install langchain langchain-openai langchain-groq --quiet

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

##2. Master prompt to define the AI model

In [3]:
MASTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """
You are a senior Coding AI and software engineering mentor.

PERSONA:
- Expert in clean code, performance optimization, and best practices
- Think critically and verify assumptions
- Explain like a professional code reviewer

ROOT BEHAVIOR RULES:
- Understand the task before responding
- Verify correctness and edge cases
- Do not hallucinate APIs or facts
- Prefer clarity, correctness, and maintainability

COGNITIVE VERIFIER PATTERN:
- Validate logic and recommendations
- Cross-check against standard Python/JavaScript best practices
- Ensure suggested changes truly improve the code

QUESTION REFINEMENT PATTERN:
- Identify missing or ambiguous information
- State assumptions explicitly
- Ask clarification questions only if they affect the outcome

PROVIDE NEW INFORMATION + ASK QUESTIONS:
- Introduce relevant best practices the user may not know
- Explain why they matter
- Ask thoughtful follow-up questions

REASONING:
- Reason internally step-by-step
- Do NOT reveal raw chain-of-thought
- Provide structured explanations only
"""),
    ("human", """
TASK:
Review the following JavaScript code for readability, performance, and best practices.

CONTEXT:
- Language: JavaScript
- Audience: Intermediate developers
- Focus: Code quality, efficiency, maintainability

Code:
{code}

OUTPUT REQUIREMENTS:
1. High-level assessment
2. Issues identified
3. Suggested improvements with justification
4. Refactored code (if appropriate)
5. New insights or best practices
6. Follow-up questions
""")
])

##3. Importing the AI model

In [4]:
from google.colab import userdata

groq_api_key = userdata.get('GROQ_API_KEY')

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=groq_api_key
)

In [5]:
chain = MASTER_PROMPT | llm

##4. Example code

In [8]:
java_code = """
import java.util.ArrayList;
import java.util.List;

public class DuplicateFinder {

    public static List<Integer> findDuplicates(int[] arr) {
        List<Integer> duplicates = new ArrayList<>();

        for (int i = 0; i < arr.length; i++) {
            for (int j = i + 1; j < arr.length; j++) {
                if (arr[i] == arr[j]) {
                    duplicates.add(arr[i]);
                }
            }
        }
        return duplicates;
    }
}
"""

##5. Response from the model

In [9]:
response = chain.invoke({
    "code": java_code
})

print(response.content)

## 1. High‑level assessment  

| Aspect | Current state | Verdict |
|--------|---------------|---------|
| **Correctness** | Finds *all* pairs of equal elements and adds the value for each pair. | Functionally works, but the result contains many repeated entries (e.g., for three identical numbers you get three copies). |
| **Readability** | Straight‑forward nested loops, but variable names are generic (`i`, `j`) and the intent isn’t documented. | Acceptable for a beginner, but can be improved with clearer naming and comments. |
| **Performance** | **O(n²)** time because of the double loop; **O(k)** extra space where *k* is the number of duplicate pairs. | Not scalable for large arrays. |
| **Maintainability** | Uses raw arrays and manual loops; any change (e.g., returning unique duplicates) requires rewriting the core logic. | Could be refactored to use the Java Collections API or Streams for easier future modifications. |
| **API design** | Returns a `List<Integer>` that may contain t